<a href="https://colab.research.google.com/github/shashank-294521/yt_rag/blob/main/YT_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip uninstall -y google-generativeai langchain-google-genai

In [ ]:
!pip install -U google-generativeai langchain-google-genai

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


indexing


In [ ]:
!pip uninstall -y youtube-transcript-api

In [ ]:
!pip install -q youtube-transcript-api==0.6.2

In [ ]:
!pip install -q yt-dlp

In [ ]:
import yt_dlp

url = "https://www.youtube.com/watch?v=Gfr50f6ZBvo&t=41s"

ydl_opts = {
    'writesubtitles': True,
    'writeautomaticsub': True,
    'skip_download': True,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

In [ ]:
subtitle_file = "Let's build GPT： from scratch, in code, spelled out. [kCc8FmEb1nY].en.vtt"

with open(subtitle_file, "r", encoding="utf-8") as f:
    transcript_text = f.read()

print(transcript_text[:2000])


In [ ]:
import re

def clean_vtt(vtt_text):
    lines = vtt_text.split("\n")

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove timestamps
        if "-->" in line:
            continue

        # Remove WEBVTT header
        if line.startswith("WEBVTT"):
            continue

        # Remove empty lines
        if line == "":
            continue

        # Remove numeric lines
        if line.isdigit():
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)

clean_transcript = clean_vtt(transcript_text)

print(clean_transcript[:2000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([clean_transcript])

print(len(chunks))

In [ ]:
chunks

text splitter


indexing (embedding generation and storing in vector store)

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embedding
)

In [ ]:
vector_store.index_to_docstore_id

Retriver


In [ ]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("what is deepmind")

Argumentation


In [ ]:
!pip install -q langchain-google-genai google-generativeai

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "enter your grok api key here"

In [ ]:
!pip install -q langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful AI assistant.

Answer ONLY from the provided context.

If answer is not found in context, say:
"I could not find this information in the video."

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)

In [ ]:
question ="is the topic on narendra modi is disscused in this video"
retrieved_docs=retriever.invoke(question)

In [ ]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt=prompt.invoke({"context":context_text,"question":question})

Generation

In [ ]:
answer = llm.invoke(final_prompt.to_string())

In [ ]:
print(answer.content)

building a chain


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
#


In [ ]:
def format_docs(retrived_docs):
  context_text="\n\n".join(doc.page_content for doc in retrived_docs)
  return context_text


In [ ]:
parallel_chain=RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is demis')

In [ ]:
parser=StrOutputParser()

In [ ]:
main_chain=parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("can u summurize the vedio")